# ShiftGuard-SecLM-1B: Primary Research Model Training Pipeline

**ShiftGuard-SecLM-1B** is a ~1 Billion parameter decoder-only causal Transformer trained strictly **from scratch with random weight initialization** (no pretrained LLM backbone, no LoRA/QLoRA).

### Core Architecture Specifications:
- **Model**: `ShiftGuard-SecLM-1B` (`configs/model/research_1b.yaml`)
- **Total Parameters**: **993,609,728 (~993.61M)** with tied embeddings
- **Dimensions**: $d_{\text{model}}=2048$, $n_{\text{layers}}=20$, $d_{\text{ffn}}=5504$
- **Attention**: Grouped-Query Attention (GQA) 2:1 ratio ($n_{\text{heads}}=16, n_{\text{kv\_heads}}=8, d_{\text{head}}=128$)
- **Normalization & Positional**: Pre-RMSNorm ($\epsilon=10^{-5}$), RoPE ($\theta=10000.0$)
- **Activation**: SwiGLU
- **VRAM Footprint**: ~7.78 GB with 8-bit AdamW + Gradient Checkpointing (fits 16GB T4/P100)
- **Author**: Masruf Rahman

In [ ]:
# 1. Hardware Inspection & VRAM Verification
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# 2. Install Required Dependencies
!pip install -q tokenizers safetensors pyyaml accelerate bitsandbytes rich tqdm

In [ ]:
# 3. Compute Feasibility & Parameter Inspection
!python scripts/estimate_compute.py

In [ ]:
# 4. Detect Existing Checkpoints for Kaggle Multi-Session Resumption
import os
from pathlib import Path

checkpoint_sources = [
    Path("/kaggle/input/shiftguard-seclm-1b-checkpoint"),
    Path("/kaggle/working/checkpoints/1b_model"),
]

latest_checkpoint = None
for src in checkpoint_sources:
    if src.exists():
        steps = sorted(src.glob("step_*"))
        if steps:
            latest_checkpoint = steps[-1]

resume_cmd = ""
if latest_checkpoint:
    print(f"[RESUME DETECTED] Found checkpoint: {latest_checkpoint}")
    resume_cmd = f"--resume-from {latest_checkpoint}"
else:
    print("[FRESH START] No prior checkpoint found. Initializing ShiftGuard-SecLM-1B from scratch.")

In [ ]:
# 5. Launch Training (ShiftGuard-SecLM-1B from scratch)
cmd = f"""python scripts/train.py \
    --config configs/model/research_1b.yaml \
    --train-data datasets/manifests/train.jsonl \
    --tokenizer-path datasets/processed/tokenizer/tokenizer.json \
    --output-dir /kaggle/working/checkpoints/1b_model \
    --epochs 5 \
    --batch-size 2 \
    --lr 2e-4 \
    --device cuda {resume_cmd}"""

print(f"Executing: {cmd}")
!{cmd}

In [ ]:
# 6. Verify Output Checkpoint & Export Metadata
import json
from pathlib import Path

out_dir = Path("/kaggle/working/checkpoints/1b_model")
if out_dir.exists():
    ckpts = sorted(out_dir.glob("step_*"))
    print(f"Found {len(ckpts)} checkpoint(s):")
    for c in ckpts:
        meta_file = c / "metadata.json"
        if meta_file.exists():
            with open(meta_file) as f:
                meta = json.load(f)
            print(f"  * {c.name}: Epoch {meta.get('epoch')}, Step {meta.get('step')}, Loss: {meta.get('loss')}")
else:
    print("No checkpoints directory found yet.")